## BASE

Forecast-based simulation with the cleaned pipeline modules.

In [1]:
path1 = 'E:/yjz/Extension for hts/JayCode/Models/'

import warnings
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
from inventory_pipeline import run_base_loop, save_pickle

In [2]:
fcst = pd.read_pickle(f"{path1}721fcsts_base2cohe.pkl").reset_index(drop=True)
fitt = pd.read_pickle(f"{path1}721fitts_base2cohe.pkl").reset_index(drop=True)

test = pd.read_pickle(f"{path1}721future_28.pkl").reset_index(drop=True)
truth = test['y'].reset_index(drop=True)
history = pd.read_pickle(f"{path1}721past_1913.pkl").reset_index(drop=True)

etsf = fitt.iloc[:, 4:].reset_index(drop=True)
lgbf = fitt.iloc[:, :4].reset_index(drop=True)

ets_resid = pd.concat({etsf.columns[i]: history['y'] - etsf.iloc[:, i] for i in range(4)}, axis=1).reset_index(drop=True)
lgb_resid = pd.concat({lgbf.columns[i]: history['y'] - lgbf.iloc[:, i] for i in range(4)}, axis=1).reset_index(drop=True)

LGB_NAMES = ['lgb_base', 'lgb_bu', 'lgb_td', 'lgb_mint']
ETS_NAMES = ['ets_base', 'ets_bu', 'ets_td', 'ets_mint']

In [3]:
lead_time = 1

Invt_df = []
for name in LGB_NAMES:
    Invt_df.append(run_base_loop(fcst=fcst[name], truth=truth, residual=lgb_resid[name], NAME=name, L_=lead_time))
lgbInvtSim = pd.concat(Invt_df, ignore_index=True)
save_pickle(lgbInvtSim, f"lgbInvtSim_L{lead_time}.pkl")

Invt_df = []
for name in ETS_NAMES:
    Invt_df.append(run_base_loop(fcst=fcst[name], truth=truth, residual=ets_resid[name], NAME=name, L_=lead_time))
etsInvtSim = pd.concat(Invt_df, ignore_index=True)
save_pickle(etsInvtSim, f"etsInvtSim_L{lead_time}.pkl")

lgbInvtSim.head()

100%|███████████████████████████████████████████████████████████████████████████| 42686/42686 [01:37<00:00, 437.44it/s]


,name,true_demand,forecasts,sst_90,arrival_90,ot_90,wip_90,ip_90,net_90,backlog_90,...,cb_95,sst_99,arrival_99,ot_99,wip_99,ip_99,net_99,backlog_99,ch_99,cb_99
0,lgb_base,4.0,5.689794,5.648312,0.000000,5.648312,5.648312,7.338106,1.689794,0.000000,...,0.000000,10.253148,0.000000,10.253148,10.253148,11.942942,1.689794,0.0,1.689794,0.0
1,lgb_base,5.0,4.679130,5.648312,5.648312,2.989336,2.989336,5.327442,2.338106,0.000000,...,0.000000,10.253148,10.253148,2.989336,2.989336,9.932278,6.942942,0.0,6.942942,0.0
2,lgb_base,7.0,4.798928,5.648312,2.989336,5.119798,5.119798,3.447240,-1.672558,1.672558,...,1.355461,10.253148,2.989336,5.119798,5.119798,8.052076,2.932278,0.0,2.932278,0.0
3,lgb_base,1.0,5.980217,5.648312,5.119798,8.181289,8.181289,10.628529,2.447240,0.000000,...,0.000000,10.253148,5.119798,8.181289,8.181289,15.233366,7.052076,0.0,7.052076,0.0
4,lgb_base,9.0,5.048564,5.648312,8.181289,0.068346,0.068346,1.696875,1.628529,0.000000,...,0.000000,10.253148,8.181289,0.068346,0.068346,6.301712,6.233366,0.0,6.233366,0.0


In [13]:
import pandas as pd
lead_time = 1
a = pd.read_pickle(f"etsInvtSim_L{lead_time}.pkl")
# a.to_csv("")
# a.iloc[:50,:]
a.iloc[:28, :12].to_csv(f"FOODS_1_001(TOTAL)_etsInvtSim.csv", index = False)

In [14]:
a = pd.read_pickle(f"lgbInvtSim_L{lead_time}.pkl")
a.iloc[:28, :12].to_csv(f"FOODS_1_001(TOTAL)_lgbInvtSim.csv", index = False)